# 데이터베이스 다루기 : PyMysql 사용

In [ ]:
# 설치
# pip install pymysql
# pip install cryptography 

In [ ]:
# 기본적인 작업 흐름
# 1. Connection 생성 ← DB 서버 접근하기 위해 필요
# 2. 위 Connection 으로부터 Cursor 객체 생성 ← DB 다루기 위해 필요
# 3. Cursor 객체를 사용하여
#    - DDL , DML 명령 실행 -> executeXXX()
#    - SELECT 명령 실행 => fetchXXX()

# 4. Cursor, Connection 객체는 시스템 자원이기 때문에 
#    사용후에는 반드시 close() 해준다


# Connection, Cursor 생성

In [1]:
import pymysql

In [8]:
conn = pymysql.connect(
    host='localhost',  # 접속url
    # port=3306,     # 디폴트 3306
    user='user2604',
    password='1234',
    database='db2604',
)


print(type(conn))
# connection 생성후 Cursor 객체 생성
cursor = conn.cursor()
print(type(cursor))

# Cursor 자원 반납
cursor.close()

# conneciton 자원 반납
conn.close() 

<class 'pymysql.connections.Connection'>
<class 'pymysql.cursors.Cursor'>


## with 구문 사용시

In [ ]:
# pymysql은 기본적으로 connection 객체는 context manager(with)를 지원하지만, 
# cursor는 구버전에서는 직접 지원이 없을 수 있다. (이 경우 보통 contextlib.closing을 같이 사용)


In [11]:
# 접속 정보 dict 로 준비
conn_info = {
    "host": "localhost",
    "user": "user2604",
    "password": '1234',
    "database": 'db2604',
}

with pymysql.connect(**conn_info) as conn:
    with conn.cursor() as cursor:
        print(type(conn))
        print(type(cursor))
        # DB 관련 쿼리 실행...

<class 'pymysql.connections.Connection'>
<class 'pymysql.cursors.Cursor'>


# DDL 데이터베이스 생성

In [37]:
conn_info = {
    "host": "localhost",
    "user": "root",
    "password": '1234',
}

try:
    with pymysql.connect(**conn_info) as conn:
        with conn.cursor() as cursor:
            sql = 'CREATE DATABASE IF NOT EXISTS pydb'
            cursor.execute(sql)   # DDL 실행 execute()

except Exception as e:
    print("💥Error:", e)

# 테이블 있는 상태에서 실행하면 에러 나온다.
# 이때 에러는 ProgrammingError 이고
# 에러 메세지는 (1007, "Can't create database 'pydb'; database exists")
# 에러 메세지는 (MySQL에러코드, MySQL에러메세지) tuple 이다


# DDL 테이블 생성

In [51]:
conn_info = {
    "host": "localhost",
    "user": "root",
    "password": '1234',
    "database": 'pydb' # database 지정
}

try:
    with pymysql.connect(**conn_info) as conn:
        with conn.cursor() as cursor:
            sql = """
            CREATE TABLE IF NOT EXISTS users (
                id INT AUTO_INCREMENT PRIMARY KEY,
                name VARCHAR(255) NOT NULL,
                age INT NOT NULL,
                created_at DATETIME NOT NULL
            ) ENGINE=InnoDB DEFAULT CHARSET=utf8            
            """
            result = cursor.execute(sql)   # DDL 실행 execute()
            print('✅테이블 생성 성공', result)  # 성공하면 0 리턴..

except Exception as e:
    print("💥Error:", e)


✅테이블 생성 성공 0


# DML: INSERT

In [57]:
from datetime import datetime, timedelta

try:
    with pymysql.connect(**conn_info) as conn:
        with conn.cursor() as cursor:
            # 방법 1
            # sql = "INSERT INTO users (name, age, created_at) VALUES ('모카', 3, '2026-01-23-13:21:56')"
            # result = cursor.execute(sql)   # DML -> 정수 리턴
            # print(f'✅ {result} 개 row insert 성공')

            # 방법 2 Prepared statement 방식
            sql = "INSERT INTO users (name, age, created_at) VALUES (%s, %s, %s)"
            # result = cursor.execute(sql, ('야옹이', 19, datetime(2004, 5, 8, 21, 19, 12)))
            # print(f'✅ {result} 개 row insert 성공')

            # 여러개 insert 가능
            now = datetime.now()
            data = [
                ("두부", 5, now + timedelta(days=5)),
                ("로켓", 21, now - timedelta(days=2)),
                ("몬지", 2, now + timedelta(hours=1)),
            ]

            result = cursor.executemany(sql, data)
            print(f'✅ {result} 개 row insert 성공')

            
        conn.commit() # DML 수행 후 commit 하기

except Exception as e:
    print("💥Error:", e)

✅ 3 개 row insert 성공
